In [6]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import struct

def save_vectors_binary(path, items):
    """
    items = [
      {
        "id": int,
        "v_core": np.ndarray,
        "v_desc": np.ndarray
      }
    ]
    """
    with open("Vector/vectors.txt", "w") as f:
        f.write(str(len(items)) + "\n")   # số item

        for it in items:
            f.write(str(it["id"]) + "\n")
            f.write(" ".join(str(x) for x in it["v_core"]) + "\n")
            f.write(" ".join(str(x) for x in it["v_desc"]) + "\n")

def main():
    model = SentenceTransformer(
        "intfloat/multilingual-e5-base"
    )
    data_names = ["drinks", "foods"]
    items = []

    for name in data_names:
        df = pd.read_json(f"Data/products_{name}.json")

        print("Đang tạo vectors...")
        
        vectors_search = []
        for i in range(df.shape[0]):
            vector = model.encode("passage: " + df.iloc[i]["name"], normalize_embeddings=True)
            vectors_search.append(vector)
            print(f"  v_core: {i+1}/{df.shape[0]}", end="\r")
        print()

        vectors_desc = []
        for i in range(df.shape[0]):
            desc_text = f"""
            {df.iloc[i]["name"]}.
            Thành phần: {df.iloc[i]["ingredients"]}.
            """
            vector = model.encode("passage: " + desc_text, normalize_embeddings=True)
            vectors_desc.append(vector)
            print(f"  v_desc: {i+1}/{df.shape[0]}", end="\r")

        for i in range(len(vectors_search)):
            items.append({
                "id": i,
                "type": 0 if name == "drinks" else 1,
                "v_core": vectors_search[i],
                "v_desc": vectors_desc[i]
            })
        
    save_vectors_binary("Vector/vectors.txt", items)
    print(vectors_desc[0])
    print(f"Đã lưu {len(items)} vectors vào vectors.txt")

if __name__ == "__main__":
    main()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 974.77it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang tạo vectors...
  v_core: 21/21
Đang tạo vectors...
  v_core: 54/54
[-7.26133538e-03  7.02470616e-02 -4.38600555e-02  3.18173096e-02
  4.46783118e-02 -3.14321294e-02 -7.31670782e-02 -5.51746339e-02
  4.56922874e-02  1.85001995e-02 -2.17314903e-03  4.06682603e-02
  1.69258729e-01  1.37665095e-02 -3.37206647e-02 -1.21471277e-02
 -3.70373297e-03 -3.82880843e-03  3.40370461e-02  2.74401810e-02
  5.76503538e-02 -4.88096960e-02 -4.98337578e-03  3.99794430e-03
  7.65095092e-03 -1.98758654e-02  4.22331737e-03  4.89735194e-02
 -5.47953360e-02  4.53766920e-02  2.78079305e-02 -3.01146526e-02
 -1.28693692e-02  2.73266006e-02  3.77475098e-02  5.85249327e-02
 -4.51307893e-02 -2.18674238e-03  3.69596817e-02 -2.42647175e-02
 -1.69899091e-02  2.88714524e-02  2.12963801e-02 -4.89621907e-02
  4.27318551e-03 -3.74104530e-02  4.17210720e-02 -8.45825393e-03
 -6.66522747e-03 -1.29353497e-02  3.01020592e-02  9.52579826e-03
  3.69011760e-02  4.36103940e-02 -4.85947728e-03 -6.18925784e-03
  2.99449991e-02  

In [14]:
import numpy as np

def read_vectors_binary(path):
    """
    Đọc vectors từ file text
    Trả về list of dict: [{"id": int,"type": int, "v_core": np.ndarray, "v_desc": np.ndarray}, ...]
    """
    items = []
    with open(path, "r") as f:
        num_items = int(f.readline().strip())
        
        print(f"Số lượng items: {num_items}")
        print("=" * 50)
        
        for _ in range(num_items):
            item_id = int(f.readline())
            
            v_core_str = f.readline()
            v_core = np.fromstring(v_core_str, sep=' ')
            
            v_desc_str = f.readline().strip()
            v_desc = np.fromstring(v_desc_str, sep=' ')
            
            items.append({
                "id": item_id,
                "v_core": v_core,
                "v_desc": v_desc
            })
    
    return items

items = read_vectors_binary("Vector/vectors.txt")

print("\n1 items đầu tiên:")
for item in items[:1]:
    print(f"\nID: {item['id']}")
    print(f"v_core shape: {item['v_core'].shape}, V_core đầu: \n{item['v_core']}")
    print(f"v_desc shape: {item['v_desc'].shape}, 5 giá trị đầu: {item['v_desc'][:5]}")

print("\n" + "=" * 50)
print("Kiểm tra normalization:")
print(f"Norm v_core[0]: {np.linalg.norm(items[0]['v_core']):.4f}")
print(f"Norm v_desc[0]: {np.linalg.norm(items[0]['v_desc']):.4f}")


Số lượng items: 75

1 items đầu tiên:

ID: 0
v_core shape: (768,), V_core đầu: 
[ 8.91740140e-04  1.18579810e-02 -2.72054800e-02  2.24928730e-02
  1.28386660e-02  6.41272300e-03 -2.36767270e-02 -3.46734800e-02
  2.93870530e-02  8.16998200e-03 -9.70174900e-03 -5.20685500e-03
  1.75440940e-01  2.82329490e-02 -2.93430970e-02 -5.53690400e-02
  1.30829040e-02 -1.17334680e-02  4.78337970e-02 -6.86439730e-03
  7.09429800e-02 -1.12568760e-02  4.48906650e-02 -1.57867540e-02
  3.94161460e-02 -8.41045200e-03  1.76009390e-02  5.04348900e-02
 -1.35127960e-02  3.88417060e-02  4.08639350e-02 -1.33005670e-02
 -1.73696350e-02  5.26029280e-02  4.43561000e-02  3.59354100e-02
 -4.67110760e-02 -9.63039900e-03  4.34697230e-02 -1.64522140e-02
  3.18296530e-03  3.51038870e-02  9.20623800e-03 -4.60522800e-02
  7.56161100e-03 -2.23596380e-02  4.66391330e-02  2.98602930e-02
 -3.98596560e-02  3.98787100e-03  1.03884000e-02 -1.67603500e-02
  1.89295060e-02  1.74475900e-02 -4.61550170e-02 -7.08227230e-03
  3.795623